# Step 3. 게임 메타데이터 수집

**목표**: 50개 게임의 정가 / Steam 공식 장르 / 멀티 여부 수집  
**입력**: `data/target_games.csv`  
**출력**: `data/game_metadata.csv`  
**소요 시간**: 약 1~2분 (50개 × 1.5초)

In [ ]:
import requests
import pandas as pd
import time
from tqdm.auto import tqdm

# Step 2 결과 로드
df = pd.read_csv("../data/target_games.csv")
print(f"로드 완료: {len(df)}개 게임")
df.head()

In [ ]:
def get_appdetails(appid):
    """
    Steam store API에서 게임 상세 정보 가져오기.
    반환: {steam_genres, price_usd, is_multiplayer} 또는 None
    """
    url = (
        f"https://store.steampowered.com/api/appdetails"
        f"?appids={appid}&cc=us&l=en&filters=genres,price_overview,categories"
    )
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json().get(str(appid), {})
        if not data.get("success"):
            return None
        d = data["data"]

        # Steam 공식 장르 (콤마 구분 문자열)
        genres = ",".join(g["description"] for g in d.get("genres", []))

        # 정가 (USD) — cents → dollars
        price_info = d.get("price_overview", {})
        price_usd = price_info.get("initial", 0) / 100.0

        # 멀티플레이 여부: category id 1(Multi-player) 또는 36(Online PvP) 포함 시 True
        category_ids = {str(c["id"]) for c in d.get("categories", [])}
        is_multiplayer = bool(category_ids & {"1", "36", "38"})

        return {
            "steam_genres": genres,
            "price_usd": price_usd,
            "is_multiplayer": is_multiplayer,
        }
    except Exception:
        return None

print("함수 정의 완료")

## 메타데이터 수집

In [ ]:
results = []
failed = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="메타데이터 수집"):
    meta = get_appdetails(row["appid"])
    time.sleep(1.5)

    if meta:
        results.append({"appid": row["appid"], **meta})
    else:
        failed.append(row["appid"])
        results.append({"appid": row["appid"], "steam_genres": None, "price_usd": None, "is_multiplayer": None})

print(f"\n수집 완료: {len(df) - len(failed)}개 성공 / {len(failed)}개 실패")
if failed:
    print(f"실패 appid: {failed}")

## 결과 확인

In [ ]:
meta_df = pd.DataFrame(results)
merged = df.merge(meta_df, on="appid")

print("장르별 멀티플레이 비율:")
print(merged.groupby("genre_category")["is_multiplayer"].mean().round(2))
print(f"\n가격 범위: ${merged['price_usd'].min()} ~ ${merged['price_usd'].max()}")
print(f"무료 게임(price=0): {(merged['price_usd'] == 0).sum()}개")
merged

## CSV 저장

In [ ]:
output_path = "../data/game_metadata.csv"
merged.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path}")
print(f"컬럼: {list(merged.columns)}")